# 01 — Build and test (CPU runtime)

**Runtime > Change runtime type > CPU**.

Run `00_setup.ipynb` first in this session (or re-run its clone/apt cells). This notebook configures Release CMake, builds, runs `ctest`, then a ThreadSanitizer build.

In [ ]:
CLONE_DIR = "/content/parallel-sci-calc"
DRIVE_ROOT = "/content/drive/MyDrive/parallel-sci-calc"
assert __import__("os").path.isfile(CLONE_DIR + "/CMakeLists.txt"), "FAIL: repo missing; run notebook 00"
print("OK: repo at", CLONE_DIR)

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
git pull --ff-only || true
chmod +x scripts/*.sh
bash scripts/colab_build.sh --jobs "$(nproc)" 2>&1 | tee /content/drive/MyDrive/parallel-sci-calc/logs/build_release.txt
echo "OK: Release build and ctest finished (see log if the command failed)"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
./build/psc_machine_info --out /content/drive/MyDrive/parallel-sci-calc/results/machine_info_from_binary.txt
echo "OK: psc_machine_info"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
echo "help" | ./build/psc | head -n 20
echo "OK: REPL smoke"

In [ ]:
%%bash
set -euo pipefail
cd /content/parallel-sci-calc
echo "--- ThreadSanitizer build (OpenMP+TSan may fail on some GCC/libgomp; paste the log if so) ---"
if bash scripts/colab_build.sh --tsan --jobs "$(nproc)" 2>&1 | tee /content/drive/MyDrive/parallel-sci-calc/logs/build_tsan.txt; then
  echo "OK: ThreadSanitizer ctest finished"
else
  echo "FAIL: ThreadSanitizer build or tests. Copy build_tsan.txt back to Cursor. Do not claim TSan is clean."
  exit 1
fi